In [1]:
# wymagane importy


from bokeh.io import output_notebook
from omni_drive import (
    Angle,
    Clock,
    Distance,
    DriveVector,
    OmniDrive,
    Recorder,
    Time,
)
from omni_drive.ui import InteractiveMode, LivePlot, Steer

output_notebook(hide_banner=True)

# OmniDrive API

<small>

Poniższy notebook przedstawia interfejs API do sterowania robotem OmniDrive oraz prezentuje jego możliwości.

</small>

### Konfiguracja połączenia z robotem:

In [2]:
robot = OmniDrive()

### Sposoby sterowania robotem OmniDrive

<small>

Biblioteka `omni_drive` udostępnia dwa główne paradygmaty sterowania platformą:

1. **Tryb reaktywny (mikro-kroki)** – pętla czasu rzeczywistego ze sprzężeniem zwrotnym. Pozwala na żywo reagować na odczyty czujników, płynnie regulować prędkość i realizować profile ruchu.
2. **Tryb deklaratywny (makro-kroki)** – sekwencje gotowych manewrów w trybie blokującym. Wykonujesz z góry zaplanowane ruchy, czekając na ukończenie każdego z nich przed przejściem dalej.

</small>

#### 1. Tryb reaktywny

<small>

Pętla taktowana przez zegar narzuca stały takt czasowy, samodzielnie dba o uśpienia.

</small>

In [3]:
async for t in Clock():
    # Zadajemy wektor prędkości (jazda do przodu z prędkością 0.2 m/s):
    robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))

    # Warunek czasowy – zatrzymanie po 2 sekundach:
    if t > 2.0:
        robot.stop()
        break


**Odczyt czujników:**

<small>

W pętli odpytujemy bezpośrednio właściwości robota i reagujemy na bieżąco.

</small>

In [ ]:
try:
    async for _ in Clock():
        # Odczytujemy stan czujnika bezpośrednio z robota:
        dist_front = robot.us_2.distance

        # Zatrzymanie gdy przeszkoda < 20 cm:
        if dist_front < Distance.cm(20):
            print("Wykryto przeszkodę - zatrzymanie!")
            break

        robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))
finally:
    robot.stop()


**Współdzielenie zegara:**

<small>

Wspólna instancja `Clock` dyktuje tempo próbkowania dla innych komponentów (np. `Recorder`) i synchronizuje rejestrację danych z pętlą robota.

</small>

In [ ]:
clock = Clock()

# Definiujemy źródła danych powiązane z zegarem i uruchamiamy wykres na żywo:
recorder = Recorder(clock, front=robot.us_2, left=robot.us_1)
LivePlot(recorder).show()

# Pętla sterowania korzysta z tego samego zegara:
async for t in clock:
    robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))

    if t > 3.0:
        robot.stop()
        break


#### 2. Tryb deklaratywny

<small>

Tryb deklaratywny jest zwięzły i świetnie sprawdza się przy z góry zaplanowanych manewrach. Ten tryb jest blokujący – program czeka na zakończenie każdego zadanego ruchu przed przejściem do kolejnej instrukcji:

</small>

In [ ]:
# 1. Jazda do przodu przez 2 sekundy:
await robot.drive_for(Time.s(2), DriveVector(velocity=0.2, heading=Angle.deg(0)))

# 2. Postój przez 1 sekundę:
await robot.pause_for(Time.s(1))

# 3. Obrót platformy w miejscu przez 2 sekundy (omega = prędkość kątowa):
await robot.drive_for(Time.s(2), DriveVector(velocity=0.0, heading=Angle(0), omega=0.5))


### Pełna specyfikacja API robota

<small>

Robot udostępnia bezpośredni dostęp do czujników odległości oraz parametrów pracy:

* **Czujniki ultradźwiękowe:**
  * `robot.us_1` – lewy (+90°)
  * `robot.us_2` – środkowy / przedni (0°)
  * `robot.us_3` – prawy (-90°)
* **Czujniki podczerwieni (IR):**
  * `robot.ir_1` – lewy
  * `robot.ir_2` – środkowy / przedni
  * `robot.ir_3` – prawy

*(Odczyt odległości z czujnika wykonujemy przez właściwość `.distance`)*.

Pozostała telemetria i stan robota:
* `robot.pose` – estymacja pozycji (`x`, `y`, `yaw`),
* `robot.imu_yaw` – kąt obrotu z żyroskopu IMU,
* `robot.power` – parametry zasilania (`voltage`, `current`, `percentage`),
* `robot.lidar` – chmura punktów ze skanera LiDAR,

</small>

### Układ odniesienia (robot.field_centric)

<small>

Podczas zadawania wektora prędkości (`DriveVector`) kierunek ruchu (`heading`) może być interpretowany w jednym z dwóch układów odniesienia w obu trybach:

* `robot.field_centric = True` (**Field-Centric, domyślny**) – kierunek liczony jest w globalnym układzie odniesienia (układzie świata). Kąt 0° to zawsze ten sam stały kierunek w pokoju, nawet jeśli robot obraca się wokół własnej osi.
* `robot.field_centric = False` (**Robot-Centric**) – kierunek liczony jest w lokalnym układzie platformy. Kąt 0° to zawsze kierunek, w który skierowany jest przód robota (`us_2`). Gdy robot się obraca, kierunki jazdy obracają się razem z nim.

Flagę można przełączać w dowolnym momencie w trakcie pracy.

</small>

**Układ odniesienia - świat:**

In [ ]:
# Domyślnie włączony jest tryb Field-Centric – kierunek zadany w globalnym układzie świata:
print(robot.field_centric) # True
# 1. Ruch w kierunku 0°
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))
# 2. Obrót o 90° stopni
await robot.rotate_for(Angle.deg(90), speed=0.5)
# 2. Ruch dalej w kierunku 0° POMIMO ZMIANY ZWROTU PLATFORMY
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))

**Układ odniesienia - nos robota:**

In [ ]:
# Przełączenie na tryb Robot-Centric – kierunek zadany względem przodu robota:
robot.field_centric = False
# 1. Ruch w kierunku 0°
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))
# 2. Obrót o 90° stopni
await robot.rotate_for(Angle.deg(90), speed=0.5)
# 3. Ruch w kierunku 90° ZGODNIE Z TYM GDZIE ZWRÓCONY JEST TERAZ ROBOT
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))

W zależności od tego ustawienia różnić się będzie m.in. zachowanie w ruchu złożonym. Tzn. takim, w którym robot obraca się podczas jazdy:

In [ ]:
await robot.drive_for(Time.s(1), DriveVector(velocity=0.4, heading=Angle.deg(0), omega=0.5))

![Porównanie trybów Field-Centric i Robot-Centric](assets/field_vs_robot_centric.svg)

**Zerowanie pozycji:**

<small>

W każdej chwili względną pozycję robota, która jest trzymana w pamięci programu, można zresetować:

</small>

In [ ]:
print(robot.pose.x, robot.pose.y, robot.pose.yaw)  # np. 0.45m  0.12m  35.0°

robot.reset_pose()

print(robot.pose.x, robot.pose.y, robot.pose.yaw)  # 0.0m  0.0m  0.0°

### Sterowanie interaktywne

<small>

Sterowanie robotem za pomocą widżetu:

</small>

In [ ]:
robot = OmniDrive()
Steer(robot).using(InteractiveMode())

BokehModel(render_bundle={'docs_json': {'a4912084-46c3-457a-965e-d45ee4cab577': {'version': '3.9.0', 'title': …